In [1]:
# Imports
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory

# Numpy Pandas
import numpy as np
import pandas as pd

# Helper libraries
import re
import os
import nltk
import clip
import string
import joblib
import requests
import chromadb
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from itertools import chain
from collections import Counter
from nltk.corpus import stopwords
from clip.simple_tokenizer import SimpleTokenizer

# Torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# Self imports
from src.data import Data


d:\venv\mm-prod-search-and-reco-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device to "Cuda" if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
# Get Data

FINAL_DATA_PATH = "../../data"
d_train = joblib.load(f"{FINAL_DATA_PATH}/train_data_with_all_embeddings.pkl")
columns = ["product_title", "product_description", "product_bullet_point", "product_brand", "product_color"]
d_clean_train = d_train.copy()
d_clean_train[columns] = d_train[columns].map(lambda x: " ".join(x))
d_clean_train

,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split_x,__index_level_0___x,...,image_url_y,Product_TFIDF_Emb,Product_CLIP_text_Emb,Product_CLIP_img_Emb,Product_Resnet18_Emb,query_doc_y,Query_TFIDF_Emb,Query_CLIP_text_Emb,Query_CLIP_img_Emb,Query_Resnet18_Emb
0,16,"[awnmower, tires, without, rims]",1,B075SCHMPY,us,I,1,1,train,16,...,https://m.media-amazon.com/images/I/81r5MLo6nr...,"[-0.010909188, -0.03643202, -0.062304936, 0.04...","[-0.041002784, -0.041515745, 0.012398187, 0.03...","[0.2733603, 0.41856885, 0.4188114, 0.19462182,...","[0.14103436, 0.23611966, 1.6405973, 0.3105259,...",awnmower tires without rims,"[0.019388188, -0.018219667, -0.0019405885, -0....","[-0.041002784, -0.041515745, 0.012398187, 0.03...",NaN,NaN
1,149,"[10, selfseal, envelopes, without, window]",5,B07RP3F2LW,us,E,1,1,train,149,...,https://m.media-amazon.com/images/I/71UzvOXH2K...,"[-0.020431643, -0.03368274, -0.066590235, 0.05...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092...",10 selfseal envelopes without window,"[-0.01314658, -0.020285046, -0.040524535, 0.03...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092..."
2,152,"[10, selfseal, envelopes, without, window]",5,B07GJCSF8Z,us,E,1,1,train,152,...,https://m.media-amazon.com/images/I/71fMUzp0Tw...,"[-0.019942034, -0.034505285, -0.06907129, 0.05...","[0.0018713473, 0.0692601, -0.017657224, -0.004...","[0.37662038, 1.1230483, 0.45779365, 0.11841136...","[1.4504515, 2.954139, 0.48512995, 0.683818, 0....",10 selfseal envelopes without window,"[-0.01314658, -0.020285046, -0.040524535, 0.03...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092..."
3,153,"[10, selfseal, envelopes, without, window]",5,B07GJKFC5X,us,E,1,1,train,153,...,https://m.media-amazon.com/images/I/71TSOB6rDA...,"[-0.019714558, -0.03488537, -0.06950915, 0.056...","[0.0017422803, 0.06550556, -0.032148566, -0.00...","[0.41479546, 1.0223008, 0.20820744, 0.3866008,...","[0.64831644, 1.8531966, 0.28947985, 0.7496531,...",10 selfseal envelopes without window,"[-0.01314658, -0.020285046, -0.040524535, 0.03...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092..."
4,155,"[10, selfseal, envelopes, without, window]",5,B07NZ4T2SL,us,I,1,1,train,155,...,https://m.media-amazon.com/images/I/71-UFz+k2p...,"[-0.016097536, -0.029461185, -0.059503444, 0.0...","[-0.0011218662, 0.041897345, -0.008049027, -0....","[0.57968336, 1.0968437, 0.10651816, 0.5060061,...","[0.8822335, 1.5593907, 2.2977557, 0.62160546, ...",10 selfseal envelopes without window,"[-0.01314658, -0.020285046, -0.040524535, 0.03...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092..."
5,157,"[10, selfseal, envelopes, without, window]",5,B078S5ZL5D,us,E,1,1,train,157,...,https://m.media-amazon.com/images/I/71jA-1adGm...,"[-0.015753336, -0.024747511, -0.049733028, 0.0...","[-0.021222068, 0.05799053, -0.014735183, 0.007...","[0.6358005, 0.6979723, 0.39382422, 0.14587788,...","[2.4287, 2.0667624, 1.8713211, 0.6760902, 1.14...",10 selfseal envelopes without window,"[-0.01314658, -0.020285046, -0.040524535, 0.03...","[-0.015938424, 0.05545586, -0.0008526017, -0.0...","[0.1852541, 1.3772874, 0.053049337, 0.10978509...","[0.104050025, 1.4776891, 0.07292303, 1.3592092..."
6,161,"[10, selfseal, envelopes, without, window]",5,B08V344BCQ,us,E,1,1,train,161,...,https://m.media-amazon.com/images/I/818BF235cX...,"[-0.020103395, -0.03201375, -0.06532879, 0.055...","[0.013892527, 0.07727057, -0.013598228, -0.005...","[0.7210572, 1.1306647, 0.0619

In [4]:
cols = [
    "query_id",
    "query_doc_x",
    "Query_TFIDF_Emb",
    "Query_CLIP_text_Emb",
    "Query_CLIP_img_Emb",
    "Query_Resnet18_Emb"
]
query_data = d_train[cols]
query_data = query_data.drop_duplicates(subset="query_id")

In [5]:
BASE_DATA_DIR =  "E:/Datasets/mm-prod-search-and-reco" # Path.absolute(__file__).parent
PROD_DATA_DIR = f"{BASE_DATA_DIR}/data/prod"

client = chromadb.PersistentClient(path=f'{PROD_DATA_DIR}/chromadb')

In [6]:
collection = client.get_collection(name = "products")

In [7]:
## Write MLP module to load TFIDF 10k model

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dims, activation):
        super().__init__()
        layers = []

        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(activation())
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, output_dims))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [8]:
def stack_emb(col):
    dim = col.dropna().iloc[0].shape[0]
    return np.stack([np.zeros(dim) if x is None or isinstance(x, float) else x for x in col])

In [9]:
query_train_comb_emb  = np.concatenate([stack_emb(query_data['Query_TFIDF_Emb']), stack_emb(query_data['Query_CLIP_text_Emb']), 
                                        stack_emb(query_data['Query_CLIP_img_Emb']), stack_emb(query_data['Query_Resnet18_Emb'])], axis=1)

In [10]:
INPUT_DIM =  512 + 768 + 768 + 512
OUTPUT_DIM = 512

## Choose the MLP settings 
hidden_dim = [1024]
activation_fn = nn.ReLU

final_model = MLP(INPUT_DIM, hidden_dim, OUTPUT_DIM, activation_fn).to(device)

FINAL_MODEL_PATH = "../../model"
final_model.load_state_dict(torch.load(f'{FINAL_MODEL_PATH}/final_model.pt', weights_only=True))

<All keys matched successfully>

In [11]:
complete_emb_list = []
for emb in query_train_comb_emb:
    with torch.no_grad():
        emb = torch.tensor(emb, dtype=torch.float32).squeeze(0)
        complete_emb_list.append(final_model(emb).numpy())

In [20]:
query_tfdif_embedding = list(complete_emb_list)
collection.query(query_embeddings=[query_tfdif_embedding[0].tolist()], n_results=10)

{'ids': [['B07RP3F2LW',
   'B07NZ4T2SL',
   'B07GJKFC5X',
   'B075SCHMPY',
   'B008005REQ',
   'B07GJCSF8Z',
   'B074M9FN2P',
   'B016V06OYS',
   'B078S5ZL5D',
   'B08V344BCQ']],
 'embeddings': None,
 'documents': [['bazic security self seal envelope 4 18 x 9 12 10 no window tint pattern mailing envelopes peel seal office checks invoices 30pack 1pack back to bazic our goal is to provide each customer with longlasting supplies at an affordable cost since 1998 we’ve delivered on this promise and will only continue to improve every year we’ve built our brand on integrity and quality so customers know exactly what to expect committed to values we are a valuedriven company guided by the principles of excellence through strong product design at low cost our commitment to these values is reflected in our dedication to improving current products and developing new exciting products for our consumers we thrive on imagination passion and leadership we have great products and will to continue to 